# Quantum vs. classical random bits

This notebook walks through the whole demo with the `pipeline` package. Every number below is computed by the same `pipeline` functions that build the presentation; nothing is worked out in the notebook itself.

The story has two points:

1. **Quantum computers are real and usable today.** The bits come from a real job on a real IBM Quantum processor, and we show the receipt.
2. **Quantum random bits are unpredictable in a way that an ordinary software generator is not.** Ordinary statistics can't tell the two streams apart. An attacker trying to predict them can.

**Which data?** The notebook picks the same folder that `python -m pipeline.tasks export` would: the latest committed run in `data/runs/`, or else the synthetic sample. To choose explicitly, set `QRNG_NOTEBOOK_RUN=<run id>` or `QRNG_NOTEBOOK_SAMPLE=<name>` before starting Jupyter, or run `python -m pipeline.tasks notebook --run <id>`.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib.ticker import StrMethodFormatter

from pipeline import plotstyle, runs
from pipeline.analysis import (
    analyze_bias,
    bias_direction,
    bitmap,
    entropy_shortfall,
    per_qubit_shannon_entropy,
    shannon_entropy_per_bit,
)
from pipeline.attacker import (
    BiasAttacker,
    MersenneTwisterAttacker,
    PreviousShotAttacker,
    cross_checks,
    stream_as_shots,
)
from pipeline.classical import urandom_words, words_to_bits
from pipeline.export import resolve_source
from pipeline.notebook import source_from_env

plotstyle.apply()

folder, is_sample = resolve_source(*source_from_env())
run = runs.load_run(folder)
SYNTHETIC = run.synthetic

q_bits = run.bits  # (shots, qubits); column j is physical qubit physical[j]
q_stream = q_bits.reshape(-1)  # shot-major: all qubits of shot 1, then shot 2, ...
c_stream = words_to_bits(run.words)  # each 32-bit word, most significant bit first
shots, n_qubits = q_bits.shape
physical = run.physical_qubits()


def show_banner():
    # A banner that can't be missed whenever the data is not from quantum hardware.
    if SYNTHETIC:
        display(
            {
                "text/html": plotstyle.banner_html(),
                "text/plain": f"*** {plotstyle.SYNTHETIC_LABEL} ***",
            },
            raw=True,
        )


def qubit_name(column):
    return f"Q{physical[column]}" if physical[column] is not None else f"column {column}"


def table(headers, rows):
    display(Markdown(plotstyle.markdown_table(headers, rows)))


show_banner()
print(f"Data folder: {folder.name} ({'sample data' if is_sample else 'committed run'})")
print(f"Quantum:   {shots:,} shots x {n_qubits} qubits = {q_stream.size:,} bits")
print(f"Classical: {run.words.size:,} words x 32 = {c_stream.size:,} bits")

## 1. Proof of hardware

We asked a real IBM Quantum processor to do the simplest possible quantum experiment, many times over: put each qubit into an equal mix of 0 and 1, then look at it. Looking forces it to pick one, and nothing decides which. Below is the receipt from IBM: the machine, the job ID, when it ran, and exactly which physical qubits were used.

Each qubit starts in $|0\rangle$. A Hadamard gate $H$ turns it into an equal superposition, and measuring it gives 1 with probability $1/2$:

$$H|0\rangle = \tfrac{1}{\sqrt{2}}\big(|0\rangle + |1\rangle\big), \qquad P(1) = \big|\langle 1|H|0\rangle\big|^2 = \tfrac{1}{2}.$$

One job runs this circuit on $N$ qubits at once for $n$ shots, giving an $n \times N$ array of bits.

In [ ]:
show_banner()
q_meta = run.quantum_meta
backend = q_meta.get("backend") or {}
job = q_meta.get("job") or {}
selection = q_meta.get("qubit_selection") or {}
sampler = q_meta.get("sampler") or {}


def shown(value):
    return "not recorded" if value is None else value


table(
    ["", ""],
    [
        ("Source", shown(q_meta.get("source"))),
        (
            "Backend",
            f"{backend['name']} ({backend.get('num_qubits')} qubits)" if backend else "none",
        ),
        ("Job ID", shown(job.get("job_id"))),
        ("Submitted (UTC)", shown(job.get("submitted_utc"))),
        ("Completed (UTC)", shown(job.get("completed_utc"))),
        ("QPU time used (s)", shown(job.get("qpu_seconds"))),
        ("Shots x qubits", f"{shots:,} x {n_qubits}"),
        ("Qubit selection method", shown(selection.get("method"))),
        ("Candidates considered", shown(selection.get("candidates"))),
        ("Calibration used (UTC)", shown(selection.get("calibration_utc"))),
        ("Readout-error mitigation", shown(sampler.get("readout_error_mitigation"))),
        (
            "Gate / measurement twirling",
            f"{shown(sampler.get('gate_twirling'))} / {shown(sampler.get('measurement_twirling'))}",
        ),
        ("Dynamical decoupling", shown(sampler.get("dynamical_decoupling"))),
    ],
)

if selection.get("used"):
    print(
        f"Qubits were selected by lowest readout error: the {n_qubits} best of "
        f"{selection.get('candidates')} candidates on {backend.get('name')}."
    )
known = [q for q in physical if q is not None]
if known:
    print(f"Physical qubits, in column order: {known}")
else:
    print("No physical qubits: this data did not come from quantum hardware.")

### How the bits are laid out

Qiskit writes measurement results as bitstrings in **little-endian** order: in `"001"` the *rightmost* character is qubit 0. Indexing the string directly would scramble the qubits, so the pipeline converts with `BitArray.to_bool_array(order="little")`. That gives an array whose column $j$ is logical qubit $j$, which ran on physical qubit `physical[j]`. The stream is then flattened **shot by shot**: all qubits of shot 1, then all qubits of shot 2, and so on. Classical 32-bit words are unpacked **most significant bit first**.

In [ ]:
from qiskit.primitives.containers import BitArray

from pipeline.quantum import bits_from_bitarray

example_strings = ["001", "110"]
example = bits_from_bitarray(
    BitArray.from_samples(example_strings, num_bits=3), n_qubits=3, shots=2
)
print(f"Qiskit bitstrings (2 shots, 3 qubits): {example_strings}")
print("As a (shots, qubits) array; column j is qubit j:")
print(example)
print(f"Flattened shot by shot: {example.reshape(-1)}")
print(f"Classical word 0x80000001, most significant bit first: {words_to_bits([0x80000001])}")

### Is every qubit a fair coin?

Real hardware isn't perfect, so each qubit's measured fraction of ones, $\hat p_j$, drifts a little from 0.5. The two charts below show how far. On the left, the spread of $\hat p_j$ across all qubits. On the right, each qubit's $\hat p_j$ against the readout error IBM reported when the qubits were chosen.

**Why isn't it exactly 50/50? Readout bias.** A qubit in state 1 carries a little energy, and while it's being measured it can lose that energy and drop to 0, so a 1 is now and then recorded as a 0. The reverse, a 0 spontaneously jumping to 1, is much rarer. On its own, that pushes $P(1)$ slightly *below* 0.5. Other small imperfections push either way: the rotation that creates the superposition can be very slightly off, and the line the detector draws between "this was a 0" and "this was a 1" may not sit exactly in the middle. **None of this is corrected:** no mitigation, no debiasing, and no qubit is dropped. The bits are reported exactly as measured.

In [ ]:
show_banner()
bias = analyze_bias(q_bits)
readout = run.readout_errors()
has_readout = any(e is not None for e in readout)

fig, (ax_hist, ax_scatter) = plt.subplots(1, 2, figsize=(15, 5.5))
ax_hist.hist(bias.p_one, bins=20, color=plotstyle.QUANTUM, edgecolor=plotstyle.SURFACE, linewidth=2)
ax_hist.axvline(0.5, color=plotstyle.REFERENCE, linestyle="--", linewidth=1.5)
ax_hist.set_xlabel("P(1)")
ax_hist.set_ylabel("Qubits")
plotstyle.title(ax_hist, f"All {n_qubits} qubits", synthetic=SYNTHETIC)

if has_readout:
    for j in range(n_qubits):
        if readout[j] is None:
            continue
        flagged = j in bias.flagged
        ax_scatter.scatter(
            readout[j],
            bias.p_one[j],
            s=70 if flagged else 45,
            color=plotstyle.FLAG if flagged else plotstyle.QUANTUM,
            edgecolor=plotstyle.SURFACE,
            linewidth=1.5,
            zorder=3,
        )
        if flagged:
            ax_scatter.annotate(
                qubit_name(j),
                (readout[j], bias.p_one[j]),
                xytext=(6, 4),
                textcoords="offset points",
                fontsize=13,
                color=plotstyle.INK,
            )
    plotstyle.reference_line(ax_scatter, 0.5, "0.5")
    ax_scatter.set_xlabel("Readout error at selection")
    ax_scatter.set_ylabel("P(1)")
    plotstyle.title(ax_scatter, f"Red: |z| > {bias.z_threshold:g}", synthetic=SYNTHETIC)
else:
    ax_scatter.axis("off")
    ax_scatter.text(
        0.5,
        0.5,
        "No readout errors:\nthis data has no hardware calibration",
        ha="center",
        va="center",
        fontsize=16,
        color=plotstyle.MUTED,
        transform=ax_scatter.transAxes,
    )
fig.tight_layout()
plt.show()

### Bias tests

Each test asks whether the differences from 0.5 are bigger than chance would produce. With $n$ shots per qubit and $N$ qubits:

- **Per-qubit z-score.** How many standard errors each qubit sits from 0.5. Qubits with $|z_j| > 3$ are flagged.
$$z_j = \frac{\hat p_j - 0.5}{\sqrt{0.25 / n}}$$
- **Spread test (chi-square).** Do the qubits differ from *each other* more than shot noise allows? Under "every qubit has the same $P(1) = \bar p$", this is chi-square with $N - 1$ degrees of freedom.
$$\chi^2 = \frac{n \sum_j (\hat p_j - \bar p)^2}{\bar p (1 - \bar p)}$$
- **Overall mean, two ways.** Is the average $\bar p$ different from 0.5? The *pooled z-test* treats all $nN$ bits as one coin, which assumes every qubit shares one $P(1)$. The *t-test across qubits* treats the $N$ values $\hat p_j$ as the sample, so it stays honest when qubits differ.
$$z = \frac{\bar p - 0.5}{\sqrt{0.25 / (nN)}}, \qquad t = \frac{\bar p - 0.5}{s_{\hat p} / \sqrt{N}}$$

A small p-value means chance alone is an unlikely explanation. When qubits really do differ, the pooled z-test overstates its case and the t-test is the one to trust.

In [ ]:
show_banner()
flagged_names = ", ".join(qubit_name(j) for j in bias.flagged) or "none"
table(
    ["Test", "Result"],
    [
        (
            f"Qubits with abs(z) > {bias.z_threshold:g}",
            f"{len(bias.flagged)} of {n_qubits}: {flagged_names}",
        ),
        (
            "Spread across qubits (chi-square)",
            f"chi-square = {bias.chi_square:.1f} on {bias.chi_square_dof} dof, "
            f"p {plotstyle.format_p(bias.chi_square_p)}",
        ),
        (
            "Overall mean, pooled z-test",
            f"mean P(1) = {bias.mean_p_one:.5f}, z = {bias.mean_z:.2f}, "
            f"p {plotstyle.format_p(bias.mean_p)}",
        ),
        (
            "Overall mean, t-test across qubits",
            f"t = {bias.across_qubits_t:.2f}, p {plotstyle.format_p(bias.across_qubits_p)}",
        ),
    ],
)

**Flagged qubits were kept, not removed.** Flagging only reports which qubits lean the most. Every qubit's bits stay in the stream and in every number that follows, because quietly dropping the "bad" ones would make the hardware look better than it is.

The chart below shows every qubit's z-score in column order, with the $\pm 3$ flag lines.

In [ ]:
show_banner()
fig, ax = plt.subplots(figsize=(15, 5))
colors = [plotstyle.FLAG if j in bias.flagged else plotstyle.QUANTUM for j in range(n_qubits)]
ax.bar(range(n_qubits), bias.z_scores, color=colors, width=0.8)
for y in (bias.z_threshold, -bias.z_threshold):
    ax.axhline(y, color=plotstyle.REFERENCE, linestyle="--", linewidth=1.5)
ax.axhline(0, color=plotstyle.MUTED, linewidth=1)
ax.set_xlabel("Column (logical qubit)")
ax.set_ylabel("z against 0.5")
plotstyle.title(
    ax, f"Red: flagged and kept ({len(bias.flagged)} of {n_qubits})", synthetic=SYNTHETIC
)
fig.tight_layout()
plt.show()

## 2. Do they look random?

Now put the quantum bits next to bits from Python's built-in `random` module (the Mersenne Twister). Both pictures below show the first 16,384 bits of each stream as a 128 x 128 grid, one pixel per bit, coloured where the bit is 1. Neither has any visible pattern.

The standard measure of "how random does this look" is **Shannon entropy**. For a stream in which a fraction $\hat p$ of bits are 1, it is

$$h(\hat p) = -\hat p \log_2 \hat p - (1 - \hat p) \log_2 (1 - \hat p)$$

bits of information per bit. A perfectly balanced stream scores 1, and any lean toward 0 or 1 lowers it. For the quantum stream we report it two ways: **pooled** over all bits, and as the **mean of the per-qubit values** $h(\hat p_j)$. The per-qubit figure is the honest one, because pooling a qubit that leans toward 0 with one that leans toward 1 cancels out both leans.

In [ ]:
show_banner()
size = 128
backend_name = backend.get("name") or "synthetic"
fig, axes = plt.subplots(1, 2, figsize=(13, 7))
for ax, stream, color, name in (
    (axes[0], c_stream, plotstyle.CLASSICAL, "Classical: Mersenne Twister"),
    (axes[1], q_stream, plotstyle.QUANTUM, f"Quantum: {backend_name}"),
):
    ax.imshow(
        bitmap(stream, size),
        cmap=plotstyle.bitmap_cmap(color),
        vmin=0,
        vmax=1,
        interpolation="nearest",
    )
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    plotstyle.title(ax, name, synthetic=SYNTHETIC)
fig.tight_layout()
plt.show()

In [ ]:
show_banner()
q_entropy = per_qubit_shannon_entropy(q_bits)
shortfall = entropy_shortfall(q_bits)
table(
    [
        "Stream",
        "Bits",
        "Fraction of ones",
        "Shannon entropy, pooled",
        "Per-qubit mean",
        "Per-qubit lowest",
    ],
    [
        (
            "Classical (Mersenne Twister)",
            f"{c_stream.size:,}",
            f"{c_stream.mean():.5f}",
            f"{shannon_entropy_per_bit(c_stream):.5f}",
            "",
            "",
        ),
        (
            f"Quantum ({backend_name})",
            f"{q_stream.size:,}",
            f"{q_stream.mean():.5f}",
            f"{shannon_entropy_per_bit(q_stream):.5f}",
            f"{q_entropy.mean():.5f}",
            f"{q_entropy.min():.5f}",
        ),
    ],
)

### Why the per-qubit estimate reads slightly low

Even a perfect coin, measured a finite number of times, comes out slightly unbalanced by chance, and any imbalance lowers $h$. So $h(\hat p)$ is biased *downward*. Expanding $h$ around 0.5, and using $\mathbb{E}[(\hat p - 0.5)^2] = 1/(4n)$ for a fair coin flipped $n$ times:

$$h(\hat p) \approx 1 - \frac{2(\hat p - 0.5)^2}{\ln 2} \quad\Longrightarrow\quad \mathbb{E}[h(\hat p)] \approx 1 - \frac{1}{2n \ln 2}.$$

Each qubit has only $n$ shots, so its estimate carries this small bias. The pooled estimate uses $nN$ bits and carries almost none. To decide whether the quantum stream really sits below 1, compare its shortfall with this expected bias. For fair coins, $\sum_j z_j^2$ follows a chi-square distribution with $N$ degrees of freedom, so that sum is the test.

In [ ]:
show_banner()
direction = bias_direction(bias.p_one)
print(f"Per-qubit mean Shannon entropy:      {shortfall.mean_entropy:.5f} bits")
print(f"Shortfall below 1:                   {shortfall.shortfall:.5f} bits")
print(
    f"Expected from estimator bias alone:  {shortfall.estimator_bias:.5f} bits"
    f"  (1 / (2 n ln 2), n = {shots:,})"
)
print(
    f"Fair-coin test: sum of z^2 = {shortfall.fair_chi_square:.1f} "
    f"on {shortfall.fair_chi_square_dof} dof, p {plotstyle.format_p(shortfall.fair_chi_square_p)}"
)

if shortfall.noticeable:
    lean = {
        "toward_0": (
            f"In this data {direction.n_below_half} of {n_qubits} qubits read 1 less than half the "
            "time, which is the direction decay during measurement predicts."
        ),
        "toward_1": (
            f"In this data {direction.n_above_half} of {n_qubits} qubits read 1 *more* than half "
            "the time, so decay during measurement is not the main effect here. The other small "
            "imperfections, which can push either way, dominate."
        ),
        "mixed": (
            f"In this data {direction.n_below_half} qubits lean toward 0 and "
            f"{direction.n_above_half} lean toward 1, so decay alone doesn't explain it. The "
            "other small imperfections, which can push either way, matter as much."
        ),
    }[direction.direction]
    cause = (
        "This synthetic sample was generated with a built-in lean to imitate readout bias, "
        "which on real hardware works like this:"
        if SYNTHETIC
        else "This is readout bias."
    )
    display(
        Markdown(
            "**The quantum stream's entropy is below 1 by more than estimator bias explains.** "
            f"{cause} A qubit in state 1 can lose its energy and decay to 0 while it is "
            "being measured, so a 1 is sometimes recorded as a 0, "
            "but a 0 almost never turns into a 1. "
            f"{lean} The bits are not corrected for this, and this is why the quantum stream's "
            "Shannon entropy can come out *lower* than the classical stream's. The difference "
            "between the two sources is not here: it is in predictability (Section 3)."
        )
    )
else:
    display(
        Markdown(
            "**The quantum stream's shortfall below 1 is within what estimator bias and chance "
            "explain.** By Shannon entropy the two streams can't be told apart."
        )
    )

## 3. The attacker

Statistics say both streams look random. The real question is whether **someone who has seen part of a stream can predict the rest**. We give each stream an attacker and score it only on bits it has never seen.

- **Classical: state recovery.** Python's `random` keeps its entire internal state in 624 numbers, and every output is a scrambled but *reversible* copy of one of them. The attacker reads 624 consecutive outputs, unscrambles them, rebuilds the state, and then runs the generator forward, predicting every later bit.
- **Quantum: bias.** No hidden state decides a measurement, so the best this attacker can do is learn each qubit's lean from the first half of the shots and then always guess that qubit's more common value.

The attacker's accuracy on held-out bits is $P_{\text{guess}}$. The headline number is **min-entropy** against that attacker, in bits of unpredictability per bit:

$$H_\infty = -\log_2 \max\big(P_{\text{guess}},\ 1 - P_{\text{guess}}\big).$$

An attacker who is reliably wrong is as useful as one who is reliably right (flip every guess), hence the max. $H_\infty = 0$ means fully predictable and $H_\infty = 1$ means a coin flip.

**Uncertainty.** $P_{\text{guess}}$ is measured on a finite number of bits, so it gets a 95% Wilson interval: with $k$ correct out of $m$ and $z = 1.96$,

$$\frac{\hat p + \frac{z^2}{2m} \pm z\sqrt{\frac{\hat p(1-\hat p)}{m} + \frac{z^2}{4m^2}}}{1 + \frac{z^2}{m}}, \qquad \hat p = k/m.$$

The **conservative** $H_\infty$ uses whichever end of the interval makes the attacker look strongest, which is the lower $H_\infty$.

In [ ]:
show_banner()
mt_attack = MersenneTwisterAttacker().attack(run.words)
bias_attack = BiasAttacker().attack(q_bits)

fig, ax = plt.subplots(figsize=(13, 6))
for result, color, name in (
    (mt_attack, plotstyle.CLASSICAL, "Classical: state recovery"),
    (bias_attack, plotstyle.QUANTUM, "Quantum: bias attacker"),
):
    ax.plot(result.running_index, result.running_accuracy, color=color, label=name)
    ax.annotate(
        f"{result.accuracy:.4f}",
        (result.running_index[-1], result.running_accuracy[-1]),
        xytext=(8, 0),
        textcoords="offset points",
        va="center",
        fontsize=14,
        color=plotstyle.INK,
        fontweight="bold",
    )
plotstyle.reference_line(ax, 0.5, "coin flip")
ax.set_ylim(0.4, 1.05)
ax.set_xlabel("Held-out bits predicted")
ax.set_ylabel("Accuracy so far")
ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.legend(loc="center right")
plotstyle.title(ax, "Predicting bits the attacker never saw", synthetic=SYNTHETIC)
fig.tight_layout()
plt.show()

In [ ]:
show_banner()
prev_attack = PreviousShotAttacker().attack(q_bits)


def attack_row(stream, result):
    return (
        result.attacker,
        stream,
        f"{result.n_training_bits:,}",
        f"{result.n_predicted:,}",
        f"{result.accuracy:.5f}",
        f"{result.ci_low:.5f} to {result.ci_high:.5f}",
        f"{result.min_entropy:.3f}",
        f"{result.min_entropy_conservative:.3f}",
    )


ATTACK_HEADERS = [
    "Attacker",
    "Stream",
    "Bits seen",
    "Bits predicted",
    "P_guess",
    "95% interval",
    "H∞ (bits)",
    "H∞ conservative",
]
table(
    ATTACK_HEADERS,
    [
        attack_row("Classical", mt_attack),
        attack_row("Quantum", bias_attack),
        attack_row("Quantum", prev_attack),
    ],
)

The third row is a **check for a cleverer attacker.** The bias attacker assumes every shot is independent. If the hardware drifted over time, or if a qubit tended to repeat (or flip) its last result, an attacker could exploit that. The previous-shot attacker learns, per qubit, whether a bit tends to repeat or flip the shot before it, and predicts each held-out bit from its predecessor. If shots are independent it should land near a coin flip whatever the bias. A $P_{\text{guess}}$ slightly below 0.5 is sampling noise, not a sign of anything.

In [ ]:
verdict = "contains" if prev_attack.consistent_with_chance else "does NOT contain"
print(
    f"Previous-shot attacker: P_guess = {prev_attack.accuracy:.5f}; its 95% interval {verdict} 0.5."
)

### Cross-checks: are the attackers rigged?

A fair objection: maybe each attacker was simply built to succeed on one stream and fail on the other. So we swap them. The state-recovery attacker is run on the quantum bits (packed into 32-bit words the same way), and the bias attacker is run on the classical bits (laid out in the same shots x qubits shape).

If the state-recovery attacker found structure in *any* stream, it would score high on the quantum bits too. If the bias attacker were simply bad, the quantum result above would mean nothing. Each one landing near 0.5 on the other stream shows that the gap between the streams comes from the **sources**, not from how the attackers were chosen.

In [ ]:
show_banner()
cross = cross_checks(q_bits, run.words)
table(
    ["Attacker", "Run on", "P_guess", "95% interval", "H∞ (bits)", "Interval contains 0.5?"],
    [
        (
            r.attacker,
            stream,
            f"{r.accuracy:.5f}",
            f"{r.ci_low:.5f} to {r.ci_high:.5f}",
            f"{r.min_entropy:.3f}",
            "yes" if r.consistent_with_chance else "no",
        )
        for stream, r in (
            ("Quantum bits", cross.mt_on_quantum),
            ("Classical bits", cross.bias_on_classical),
        )
    ],
)

## Appendix: secure classical generators

The Mersenne Twister is used here because it is what Python's default `random` module gives you, and many programs reach for it. It is **not** the best classical option. Operating systems also provide cryptographically secure generators, such as `os.urandom`. Below, the bias attacker and the state-recovery attacker are both run on fresh `os.urandom` bits, laid out exactly like the quantum stream. (These bits are drawn fresh each time the notebook runs, so the numbers change slightly between runs.)

In [ ]:
u_words = urandom_words(run.words.size)
u_bias = BiasAttacker().attack(stream_as_shots(words_to_bits(u_words), q_bits.shape))
u_mt = MersenneTwisterAttacker().attack(u_words)
table(ATTACK_HEADERS, [attack_row("os.urandom", u_bias), attack_row("os.urandom", u_mt)])

**Computational versus physical unpredictability.** Neither attacker gets anywhere with `os.urandom`, and no known practical attack does. But the reason is different from the quantum case. A secure generator is still a deterministic algorithm with a hidden internal state (fed by hardware noise in the computer). Its unpredictability rests on **computational hardness**: recovering that state from the output is believed to take more computing power than anyone has. That belief is extremely well founded, but it is an assumption about how hard certain problems are, and it would fail if the state leaked or the algorithm were broken. A quantum measurement has **no hidden state to recover**. Quantum mechanics says the outcome is not decided until it happens, so the unpredictability rests on **physics**, not on anyone's limited computing power.

**Limits of what we measured.** $H_\infty$ here is min-entropy *against these attackers*. It is not a certified, device-independent bound. A cleverer attacker might exploit drift over time or correlations between qubits. The previous-shot attacker in Section 3 checks the simplest such effect. A full guarantee would need a device-independent protocol, and that is beyond this demo.